# SIGNOVA — Incremental & Resource-Aware Training Pipeline

This notebook implements the continuous sign language training pipeline for **ArabSign (ArSL)** and **How2Sign (ASL)**.

### Features:
- **Hardware Acceleration:** Native PyTorch CUDA support on local GPU (RTX 5060) or Google Colab / Kaggle GPU.
- **Strict Linguistic Separation:** Independent vocabulary and loss computation for ArabSign and How2Sign.
- **Signer-Independent Protocol:** Zero data leakage across training, validation, and test partitions.
- **Resource-Aware:** Configurable batch sizes and sample limits to prevent RAM/VRAM saturation.

In [ ]:
import sys
from pathlib import Path
import torch
import yaml

# Ensure project root is in sys.path
repo_root = Path("..").resolve().parents[1]
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device selected for training: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    # Blackwell architecture fix for PyTorch sequence kernels
    torch.backends.cudnn.enabled = False

## 1. Load Training Configuration

In [ ]:
config_path = repo_root / "ml/sign_language/configs/config.yaml"
with open(config_path, encoding="utf-8") as f:
    config = yaml.safe_load(f)

print("Model Configuration:", config["model"])
print("Training Hyperparameters:", config["training"])

## 2. Execute Training: ArabSign (Continuous Arabic Sign Language - ArSL)

In [ ]:
from ml.sign_language.src.train import run_training

print("=== Starting ArabSign (ArSL) Training ===")
arab_results = run_training(
    dataset_name="arabsign",
    epochs=5,
    batch_size=16,
    max_samples=None,  # Full 6,323 training samples
)

print("\nArabSign Training Completed!")
print(f"Best Validation Loss: {arab_results['best_val_loss']:.4f}")
print(f"Saved Artifacts: {arab_results['artifact_dir']}")

## 3. Execute Pilot Training: How2Sign (Continuous ASL)

In [ ]:
print("=== Starting How2Sign (ASL) Pilot Training ===")
how_results = run_training(
    dataset_name="how2sign",
    epochs=5,
    batch_size=16,
    max_samples=2000,  # Pilot subset
)

print("\nHow2Sign Training Completed!")
print(f"Best Validation Loss: {how_results['best_val_loss']:.4f}")
print(f"Saved Artifacts: {how_results['artifact_dir']}")